# Line Following + Green Dot Detection

GitHub working copy based on the existing v3 line-following structure.

## Sections
1. Initialize
2. Camera frame callback
3. Black-line detection (`far`, `middle`, `near`)
4. Green-dot detection
5. Green direction from the black-line virtual direction
6. Preview
7. Existing line-follow controller
8. Green-aware turn controller
9. Preview run
10. Short driving test

> Safety: run the preview before enabling motor drive. Tune thresholds and motor values on the actual robot.


In [ ]:
import time
import cv2 #카메라
import numpy as np # 
import traitlets #값 바뀔 때 신호 연동
import ipywidgets as widgets #화면 위젯

from IPython.display import display #
from jetbot import Camera, Robot, bgr8_to_jpeg

robot = Robot()
robot.stop()

camera = Camera.instance(width=224, height=224)

live_view = widgets.Image(format="jpeg", width=336, height=336) #camera가 보여주는 영상 만들 공간

camera_link = traitlets.dlink(
    (camera, "value"),
    (live_view, "value"),
    transform=bgr8_to_jpeg
)

display(live_view)


ModuleNotFoundError: No module named 'cv2'

In [ ]:
latest = {"sample": None} #가장 최근에 카메라가 인식했던 영상 사진 

def receive_frame(change): #
    latest["sample"] = (change["new"], time.monotonic()) #새 장면 찍힐 때 마다 이미지와 시간

camera.observe(receive_frame, names="value") #카메라가 새로운 장면 보일 때 마다 리시브 프레임 실행해서 저장

time.sleep(0.5) #카메라 콜백 시간

if latest["sample"] is None:
    print("No camera frame yet.")
else:
    print("Camera frame received.")


## 3. Black-line detection

In [ ]:
BLACK_THRESHOLD = 70

BANDS = [
    ("near", 0.72, 0.88),
    ("middle", 0.54, 0.70),
    ("far", 0.36, 0.52),
]

def detect_curve(frame):
    height, width = frame.shape[:2] #사진 크기 가져오기 (대충 224x224)

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY) #컬러 -> 흑백 : 검정석 밝기만 볼 수 있음
    gray = cv2.GaussianBlur(gray, (5, 5), 0) #흐리게 만들어서 노이즈 줄이기

#마스크 해서 검정선 뽑기 -> 윤곽선 따기 ( 이 윤곽선은 뜨레쉬홀드가)
    _, mask = cv2.threshold( 
        gray, BLACK_THRESHOLD, 255, cv2.THRESH_BINARY_INV
    )

    overlay = frame.copy()
    positions = {}

#이건 그냥 박스 3개 위치 설정
    for name, top, bottom in BANDS:
        y0 = int(top * height)
        y1 = int(bottom * height)

        #각 박스끼리 따로보겠다 : 각 박스 3개중 하나
        strip = mask[y0:y1, :]

#흑백 전환했으니 흰석의 테두리를 착기 : findcontours , 가장 바깥 :retr_external
        contours, _ = cv2.findContours(
            strip, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
        )

        positions[name] = None

        if contours:
            """윤곽선 여러개 있으면 가장 큰거 선택 : 그 점선 때문에 근데 
            생각해보니 왔다리 갔다리 점선에 가장 큰 걸 그어버리면 직진할낀디? 이건 더블체크
            """
            contour = max(contours, key=cv2.contourArea)
            area = cv2.contourArea(contour)
            fraction = area / float(strip.size)

            #무게 중심 계산에 필요한 값
            moments = cv2.moments(contour)


            if 0.005 < fraction < 0.80 and moments["m00"] > 0:
                #검정선 중심 좌표들
                cx = moments["m10"] / moments["m00"]
                cy = moments["m01"] / moments["m00"] + y0

                #-1 0 1 이렇게 3개로 섹션 세로로 나누기 저 중심선 기준
                x_norm = (cx - width / 2.0) / (width / 2.0)
                positions[name] = float(x_norm)

                #이거 그 보이는 초록점 찍는 거 
                cv2.circle(
                    overlay, (int(cx), int(cy)), 4, (0, 255, 0), -1
                )

        #이건 보이는 네모 찍는거
        cv2.rectangle(
            overlay, (0, y0), (width - 1, y1), (255, 0, 0), 1
        )
        cv2.putText(
            overlay, name, (3, y0 + 14),
            cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 0, 255), 1
        )

    #네모 구분선 찍는거 위에 다 디자인 퍼포즈
    cv2.line(
        overlay, (width // 2, 0), (width // 2, height), (0, 255, 255), 1
    )

    return positions, overlay, mask


SyntaxError: invalid syntax (1720656038.py, line 44)

## 4. Green-dot detection

In [ ]:
# Starting 색상,채도,밝기 값인데 이거 튜닝해야함
GREEN_LOWER = np.array([35, 70, 50], dtype=np.uint8)
GREEN_UPPER = np.array([90, 255, 255], dtype=np.uint8)

"""
이건 초록색 최소 최대 구해서 만약 범위 밖이면 노이즈 처리인데 굳이...?
GREEN_MIN_AREA = 40
GREEN_MAX_AREA = 3000
"""

#BGR -> HSV
def detect_green(frame):
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)


    #초록색만 추출 
    green_mask = cv2.inRange(hsv, GREEN_LOWER, GREEN_UPPER)


"""
이건 초록부분 깔끔하게 만들어주는 필터인데 필요하면 키고 없으면 꺼도 괜찮아

    kernel = np.ones((5, 5), np.uint8)

    green_mask = cv2.morphologyEx(
        green_mask, cv2.MORPH_OPEN, kernel
    )
    green_mask = cv2.morphologyEx(
        green_mask, cv2.MORPH_CLOSE, kernel
    )
"""


    #초록색 마킹 찾기
    contours, _ = cv2.findContours(
        green_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )

    #그 빈공간 하나 만들어서
    markers = []
    #마지막 저장된 사진 하나 따오기
    overlay = frame.copy()


    for contour in contours:

        """ 이건 크기 이상한 초록색 거르는 거라 일단 비활성화
        area = cv2.contourArea(contour)

        if area < GREEN_MIN_AREA or area > GREEN_MAX_AREA:
            continue
        """

        """
        크기 검사 후 모양 검사 하는 필터 -=> 일단 비활
        x, y, w, h = cv2.boundingRect(contour)

        if w <= 0 or h <= 0:
            continue

        aspect = w / float(h)

        if aspect < 0.45 or aspect > 2.2:
            continue

        """

        moments = cv2.moments(contour)

        if moments["m00"] == 0:
            continue

        #초록색 부분의 중심된 부분의 좌표를 구함(초록색이 네모니까 그 네모 중심)
        cx = moments["m10"] / moments["m00"]
        cy = moments["m01"] / moments["m00"]

        #그린닷 정보 저장
        markers.append({
            "x": float(cx),
            "y": float(cy),
            "area": float(area),\

            #그린닷 감싸는 박스 크기
            "w": int(w),
            "h": int(h),
        })

        #그린닷 위에 초록박스 그리고 점찍고 텍스트 넣기
        cv2.rectangle(
            overlay, (x, y), (x + w, y + h), (0, 255, 0), 2
        )
        cv2.circle(
            overlay, (int(cx), int(cy)), 4, (0, 255, 0), -1
        )
        cv2.putText(
            overlay, "GREEN", (x, max(15, y - 5)),
            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1
        )

    """그린닷 크기 큰 순으로 재배치

    markers.sort(key=lambda marker: marker["area"], reverse=True)

    return markers, green_mask, overlay
    """
    

IndentationError: unexpected indent (3900982702.py, line 35)

## 5. Virtual black-line direction and Green LEFT/RIGHT

In [ ]:
GREEN_SIDE_MARGIN = 8.0

# far middle near 을 좌표로 변환
def get_line_points(positions, width, height):
    band_y = {
        "far": 0.44,
        "middle": 0.62,
        "near": 0.80,
    }


    points = []

    # 먼곳부터 차례대로처리
    for name in ("far", "middle", "near"):
        
        # positions 안에 저장해둔 해당 박스의 검정선 위치값을 가져오는 만약 건너뛰면 못 찾은거 none
        value = positions.get(name)
        if value is None:
            continue

        #
        x = width / 2.0 + value * width / 2.0
        y = band_y[name] * height
        points.append((float(x), float(y)))

    return points

# 어디로 가는지 
def get_virtual_direction(points):
    if len(points) < 2:
        return None

    near_point = points[-1]
    far_point = points[0]

    #y,x방향 움직임 보고 방향 벡터
    dx = far_point[0] - near_point[0]
    dy = far_point[1] - near_point[1]
    length = np.hypot(dx, dy)


    if length < 1e-6:
        return None

    return dx / length, dy / length


# 검정선 기준으로 그린닷이 어디에 있는지
def classify_green_direction(positions, markers, frame):
    if not markers: # 마커 : 그린닷
        return "NONE"

    
    #이미지 크기 가져오기
    height, width = frame.shape[:2]
    #점 3개 가져오기 
    points = get_line_points(positions, width, height)

    #점이 두개면 리턴
    if len(points) < 2:
        return "UNKNOWN"

    #점 방향 계산
    direction = get_virtual_direction(points)

    if direction is None:
        return "UNKNOWN"

    #방향이 어디를 향하고 있는지
    dx, dy = direction
    line_x, line_y = points[-1]

    directions = []

    #그린닷 하나씩 검사하기
    for marker in markers:

        #x,y방향 거리
        px = marker["x"] - line_x
        py = marker["y"] - line_y

        # Signed side of the virtual line.
        side_score = px * dy - py * dx

        #검정색 라인 기준으로 그린닷 위치 정하기
        if side_score > GREEN_SIDE_MARGIN:
            directions.append("LEFT")
        elif side_score < -GREEN_SIDE_MARGIN:
            directions.append("RIGHT")
        else:
            directions.append("CENTER")

    if "LEFT" in directions and "RIGHT" in directions:
        return "BOTH"
    if "LEFT" in directions:
        return "LEFT"
    if "RIGHT" in directions:
        return "RIGHT"

    return "CENTER"


## 6. Stationary preview

In [ ]:
overlay_view = widgets.Image(format="jpeg", width=336, height=336)
mask_view = widgets.Image(format="jpeg", width=336, height=336)
status_view = widgets.Label(value="Preview ready")

display(
    widgets.VBox([
        widgets.HBox([overlay_view, mask_view]),
        status_view
    ])
)

def preview_once():
    sample = latest["sample"]

    if sample is None:
        print("No camera frame.")
        return

    frame, _ = sample

    positions, line_overlay, _ = detect_curve(frame)
    markers, green_mask, _ = detect_green(frame)

    overlay = line_overlay.copy()

    height, width = frame.shape[:2]
    points = get_line_points(positions, width, height)

    if len(points) >= 2:
        for i in range(len(points) - 1):
            p1 = (int(points[i][0]), int(points[i][1]))
            p2 = (int(points[i + 1][0]), int(points[i + 1][1]))
            cv2.line(overlay, p1, p2, (255, 0, 255), 2)

    for point in points:
        cv2.circle(
            overlay, (int(point[0]), int(point[1])), 4, (255, 0, 255), -1
        )

    for marker in markers:
        cv2.circle(
            overlay, (int(marker["x"]), int(marker["y"])), 9, (0, 255, 0), 2
        )

    green_state = classify_green_direction(
        positions, markers, frame
    )

    cv2.putText(
        overlay, "GREEN: " + green_state, (5, 20),
        cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 255, 0), 2
    )

    overlay_view.value = bgr8_to_jpeg(overlay)
    mask_view.value = bgr8_to_jpeg(
        cv2.cvtColor(green_mask, cv2.COLOR_GRAY2BGR)
    )

    status_view.value = (
        f"far={positions.get('far')} | "
        f"middle={positions.get('middle')} | "
        f"near={positions.get('near')} | "
        f"green={green_state} | count={len(markers)}"
    )

preview_once()


## 7. Line-follow controller

In [ ]:
BASE_SPEED = 0.09
CURVE_SPEED = 0.13
KP = 0.11
MAX_SPEED = 0.25
TURN_SPEED = 0.15

TURN_ENTER = 0.55
TURN_EXIT = 0.22
TURN_TIMEOUT = 1.0
LOST_TIMEOUT = 0.35

def clamp(value, low, high):
    return max(low, min(high, value))

class LineController:
    def __init__(self):
        self.turn_direction = 0
        self.turn_started = None
        self.lost_started = None
        self.last_direction = 0
        self.stopped = False

    def stop(self, reason):
        self.stopped = True
        return 0.0, 0.0, "STOP: " + reason

    def pivot(self, direction):
        speed = clamp(TURN_SPEED, 0.0, MAX_SPEED)

        if direction > 0:
            return speed, -speed

        return -speed, speed

    def step(self, positions, now):
        if self.stopped:
            return self.stop("controller stopped")

        visible = [
            (name, value)
            for name, value in positions.items()
            if value is not None
        ]

        if not visible:
            if self.lost_started is None:
                self.lost_started = now

            if now - self.lost_started >= LOST_TIMEOUT:
                return self.stop("line not reacquired")

            if not self.last_direction:
                return self.stop(
                    "line lost without direction"
                )

            if self.turn_started is None:
                self.turn_started = now
                self.turn_direction = self.last_direction

            if now - self.turn_started >= TURN_TIMEOUT:
                return self.stop("turn timed out")

            left, right = self.pivot(self.turn_direction)
            return left, right, "SEARCH"

        self.lost_started = None

        weights = {
            "near": 0.45,
            "middle": 0.35,
            "far": 0.20
        }

        denominator = sum(
            weights[name] for name, _ in visible
        )

        error = sum(
            weights[name] * value
            for name, value in visible
        ) / denominator

        farthest = positions.get("far")

        if farthest is None:
            farthest = positions.get("middle")

        near = positions.get("near")

        if near is not None and farthest is not None:
            error += 0.40 * (farthest - near)

        error = clamp(error, -1.0, 1.0)

        if abs(error) > 0.15:
            self.last_direction = 1 if error > 0 else -1

        if self.turn_started is not None:
            if (
                near is not None
                and abs(near) < TURN_EXIT
                and abs(error) < TURN_EXIT
            ):
                self.turn_started = None
                self.turn_direction = 0

            elif now - self.turn_started >= TURN_TIMEOUT:
                return self.stop("turn timed out")

        if (
            self.turn_started is None
            and abs(error) >= TURN_ENTER
        ):
            self.turn_started = now
            self.turn_direction = 1 if error > 0 else -1

        if self.turn_started is not None:
            left, right = self.pivot(self.turn_direction)
            return left, right, "TURN"

        bend = min(abs(error) / TURN_ENTER, 1.0)

        base = (
            BASE_SPEED
            + (CURVE_SPEED - BASE_SPEED) * bend
        )

        correction = KP * error

        left = clamp(base + correction, 0.0, MAX_SPEED)
        right = clamp(base - correction, 0.0, MAX_SPEED)

        return left, right, "FOLLOW"


## 8. Green-aware turn controller

In [ ]:
GREEN_CONFIRM_FRAMES = 3
GREEN_STOP_TIME = 0.12

TURN_REACQUIRE_TIMEOUT = 1.5

UTURN_SPEED = 0.15
UTURN_TIMEOUT = 1.8

class GreenController(LineController):
    def __init__(self):
        super().__init__()

        self.green_candidate = "NONE"
        self.green_frames = 0

        self.green_stop_until = None

        self.action = "FOLLOW"
        self.action_started = None

    def start_green_action(self, direction, now):
        self.green_stop_until = now + GREEN_STOP_TIME
        self.action_started = now

        if direction == "LEFT":
            self.action = "TURN_LEFT"
        elif direction == "RIGHT":
            self.action = "TURN_RIGHT"
        elif direction == "BOTH":
            self.action = "U_TURN"
        else:
            self.action = "FOLLOW"

    def turn_command(self):
        if self.action == "TURN_LEFT":
            return -TURN_SPEED, TURN_SPEED

        if self.action == "TURN_RIGHT":
            return TURN_SPEED, -TURN_SPEED

        if self.action == "U_TURN":
            # Implementation choice: U-turn always to the left.
            return -UTURN_SPEED, UTURN_SPEED

        return 0.0, 0.0

    def step_with_green(self, positions, markers, now, frame):
        if self.action in (
            "TURN_LEFT", "TURN_RIGHT", "U_TURN"
        ):
            near = positions.get("near")
            middle = positions.get("middle")

            line_reacquired = (
                near is not None
                and abs(near) < TURN_EXIT
            )

            if (
                self.action != "U_TURN"
                and line_reacquired
                and now - self.action_started > 0.20
            ):
                self.action = "FOLLOW"
                self.action_started = None

            elif (
                self.action == "U_TURN"
                and line_reacquired
                and middle is not None
                and abs(middle) < TURN_EXIT
                and now - self.action_started > 0.35
            ):
                self.action = "FOLLOW"
                self.action_started = None

            elif (
                now - self.action_started
                >= (
                    UTURN_TIMEOUT
                    if self.action == "U_TURN"
                    else TURN_REACQUIRE_TIMEOUT
                )
            ):
                self.action = "FOLLOW"
                self.action_started = None

            else:
                return (*self.turn_command(), self.action)

        green_direction = classify_green_direction(
            positions, markers, frame
        )

        if green_direction in ("NONE", "UNKNOWN", "CENTER"):
            self.green_candidate = "NONE"
            self.green_frames = 0

        else:
            if green_direction == self.green_candidate:
                self.green_frames += 1
            else:
                self.green_candidate = green_direction
                self.green_frames = 1

            if self.green_frames >= GREEN_CONFIRM_FRAMES:
                self.start_green_action(
                    green_direction, now
                )
                self.green_candidate = "NONE"
                self.green_frames = 0

                return 0.0, 0.0, "GREEN_STOP"

        if self.green_stop_until is not None:
            if now < self.green_stop_until:
                return 0.0, 0.0, "GREEN_STOP"

            self.green_stop_until = None

        return self.step(positions, now)


## 9. Preview run

In [ ]:
def run_preview(seconds=20):
    controller = GreenController()

    deadline = time.monotonic() + seconds
    previous_stamp = None

    try:
        while time.monotonic() < deadline:
            sample = latest["sample"]

            if sample is None:
                time.sleep(0.01)
                continue

            frame, captured_at = sample

            if captured_at == previous_stamp:
                time.sleep(0.005)
                continue

            previous_stamp = captured_at

            positions, overlay, _ = detect_curve(frame)
            markers, green_mask, _ = detect_green(frame)

            now = time.monotonic()

            left, right, state = controller.step_with_green(
                positions, markers, now, frame
            )

            green_state = classify_green_direction(
                positions, markers, frame
            )

            height, width = frame.shape[:2]
            points = get_line_points(
                positions, width, height
            )

            if len(points) >= 2:
                for i in range(len(points) - 1):
                    p1 = (int(points[i][0]), int(points[i][1]))
                    p2 = (int(points[i + 1][0]), int(points[i + 1][1]))
                    cv2.line(
                        overlay, p1, p2, (255, 0, 255), 2
                    )

            for point in points:
                cv2.circle(
                    overlay,
                    (int(point[0]), int(point[1])),
                    4,
                    (255, 0, 255),
                    -1
                )

            for marker in markers:
                cv2.circle(
                    overlay,
                    (int(marker["x"]), int(marker["y"])),
                    9,
                    (0, 255, 0),
                    2
                )

            cv2.putText(
                overlay,
                "GREEN: " + green_state,
                (5, 20),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.55,
                (0, 255, 0),
                2
            )

            cv2.putText(
                overlay,
                "STATE: " + state,
                (5, 42),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.55,
                (255, 255, 255),
                2
            )

            overlay_view.value = bgr8_to_jpeg(overlay)

            mask_view.value = bgr8_to_jpeg(
                cv2.cvtColor(green_mask, cv2.COLOR_GRAY2BGR)
            )

            status_view.value = (
                f"{state} | green={green_state} | "
                f"left={left:.2f} | right={right:.2f}"
            )

            time.sleep(0.01)

    finally:
        robot.stop()
        print("Preview ended; motors stopped.")


## 10. Short driving test

Enable only after the stationary preview behaves correctly.


In [ ]:
def run_robot(seconds=2):
    controller = GreenController()

    deadline = time.monotonic() + seconds
    previous_stamp = None

    try:
        while time.monotonic() < deadline:
            sample = latest["sample"]

            if sample is None:
                robot.stop()
                time.sleep(0.01)
                continue

            frame, captured_at = sample

            if captured_at == previous_stamp:
                time.sleep(0.005)
                continue

            previous_stamp = captured_at

            positions, _, _ = detect_curve(frame)
            markers, _, _ = detect_green(frame)

            now = time.monotonic()

            left, right, state = controller.step_with_green(
                positions, markers, now, frame
            )

            robot.set_motors(left, right)

            if controller.stopped:
                print(state)
                break

            time.sleep(0.01)

    finally:
        robot.stop()
        print("Run ended; motors stopped.")

# Start with a very short test:
# run_robot(seconds=2)


## Notes for teammate

- This notebook keeps the three-strip black-line approach.
- Green markers are detected independently using HSV thresholding.
- LEFT/RIGHT is calculated relative to the estimated black-line direction from `far/middle/near`.
- A green decision is confirmed for multiple frames before a short stop.
- `BOTH` is treated as a dead-end and this implementation uses a left U-turn.
- Tune green HSV thresholds, area limits, motor speeds, stop time, and turn/reacquisition time on the real robot.
